# Optimized Thai Digit Classifier (51–55)

| Step | Detail |
|------|--------|
| **6) Preprocessing** | Grayscale → Denoise (Gaussian σ=0.8) → Centering (Otsu bbox) → Resize 28×28 → Normalize [0,1] |
| **Feature Extraction** | HOG — 9 orientations, 4×4 cells, 2×2 blocks → **1,296-d vector** |
| **Optimization** | RandomizedSearchCV (30 iter, 3-fold CV, F1-macro) |
| **7) Evaluation** | Accuracy · Precision · Recall · F1-score (macro) + Confusion Matrix |

Models tuned: Random Forest · SVM (RBF) · MLP · KNN  
Baselines (no search): Decision Tree · Naive Bayes

In [ ]:
!pip -q install scikit-learn scikit-image scipy joblib pillow matplotlib seaborn pandas
# torch / torchvision are pre-installed on Colab

## 1. Upload Dataset

In [ ]:
from google.colab import files
import zipfile, os

uploaded = files.upload()          # upload dataset.zip
zip_name = next(iter(uploaded))
with zipfile.ZipFile(zip_name) as z:
    z.extractall('.')

DATA_DIR = 'dataset'
print('Classes found:', sorted(os.listdir(DATA_DIR)))

## 2. Config & Imports

In [ ]:
import os, glob, time, warnings
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from PIL import Image
from collections import Counter
import joblib
warnings.filterwarnings('ignore')

from scipy.ndimage import gaussian_filter
from skimage.feature import hog as skimage_hog
from skimage.filters import threshold_otsu

from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import (
    train_test_split, StratifiedKFold,
    RandomizedSearchCV, cross_val_score,
)
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix,
)
from scipy.stats import randint, loguniform

IMG_SIZE      = 28
VAL_SPLIT     = 0.15
AUG_PER_IMAGE = 10
SEED          = 42
N_SEARCH_ITER = 30

CLASS_NAMES = sorted(
    d for d in os.listdir(DATA_DIR)
    if os.path.isdir(os.path.join(DATA_DIR, d))
)
assert CLASS_NAMES == ['51','52','53','54','55'], CLASS_NAMES
label_to_idx = {c: i for i, c in enumerate(CLASS_NAMES)}
print('Classes:', CLASS_NAMES)

## 3. Load Raw Images

In [ ]:
samples = []
for c in CLASS_NAMES:
    for path in glob.glob(os.path.join(DATA_DIR, c, '*')):
        try:
            img = Image.open(path).convert('L')
            samples.append((img, label_to_idx[c]))
        except Exception as e:
            print('Skipped:', path, e)

print(f'Total images: {len(samples)}')
print('Per class:', {CLASS_NAMES[i]: c for i, c in Counter(y for _, y in samples).items()})

## 6. Preprocessing Pipeline

**Required:** Grayscale → Resize → Normalize  
**Added:** Denoise (Gaussian blur σ=0.8) → Thresholding (Otsu) → Centering (bounding-box crop + 20% padding)

**HOG Features** (Histogram of Oriented Gradients) — standard descriptor for shape-based recognition.  
Captures edge directions and gradients, far more informative than raw pixels.

In [ ]:
def preprocess(pil_img, img_size=IMG_SIZE):
    """
    6) Preprocessing:
      1. Grayscale
      2. Denoise  — Gaussian blur (sigma=0.8)
      3. Threshold — Otsu method locates digit region
      4. Center   — crop to bounding box + 20% padding
      5. Resize   — img_size x img_size (BILINEAR)
      6. Normalize — to [0, 1]
    """
    arr = np.array(pil_img.convert('L'), dtype=np.float32)

    # 2. Denoise
    arr = gaussian_filter(arr, sigma=0.8)

    # 3 & 4. Threshold + Center
    try:
        thresh     = threshold_otsu(arr)
        digit_mask = arr < thresh          # digit is dark on white background
        rows = np.any(digit_mask, axis=1)
        cols = np.any(digit_mask, axis=0)
        if rows.any() and cols.any():
            r0, r1 = np.where(rows)[0][[0, -1]]
            c0, c1 = np.where(cols)[0][[0, -1]]
            side   = max(r1 - r0 + 1, c1 - c0 + 1)
            pad    = max(int(side * 0.20), 3)
            r0 = max(0, r0 - pad);   r1 = min(arr.shape[0]-1, r1 + pad)
            c0 = max(0, c0 - pad);   c1 = min(arr.shape[1]-1, c1 + pad)
            arr = arr[r0:r1+1, c0:c1+1]
    except Exception:
        pass  # blank image — skip centering

    # 5. Resize
    img_out = Image.fromarray(np.clip(arr, 0, 255).astype(np.uint8))
    img_out = img_out.resize((img_size, img_size), Image.BILINEAR)
    arr     = np.array(img_out, dtype=np.float32)

    # 6. Normalize
    return arr / 255.0


def extract_hog(arr_norm):
    """HOG: 9 orientations, 4x4 cell, 2x2 block -> 1,296-d vector."""
    return skimage_hog(
        arr_norm,
        orientations=9,
        pixels_per_cell=(4, 4),
        cells_per_block=(2, 2),
        block_norm='L2-Hys',
        visualize=False,
    ).astype(np.float32)


def to_feature_vec(pil_img):
    """PIL -> preprocess -> HOG -> 1,296-d float32."""
    return extract_hog(preprocess(pil_img))


_dim = len(to_feature_vec(Image.new('L', (64, 64), 255)))
print(f'HOG feature dimension: {_dim}  (expected 1296)')

## 4. Augmentation + Feature Extraction (10x per image)

In [ ]:
from torchvision import transforms

aug = transforms.Compose([
    transforms.RandomAffine(
        degrees=15, translate=(0.12, 0.12),
        scale=(0.75, 1.25), shear=8, fill=255,
    ),
    transforms.RandomPerspective(distortion_scale=0.2, p=0.4, fill=255),
    transforms.RandomApply(
        [transforms.ElasticTransform(alpha=20.0, sigma=4.0)], p=0.25
    ),
    transforms.RandomApply(
        [transforms.GaussianBlur(kernel_size=3, sigma=(0.5, 1.2))], p=0.2
    ),
])

X_list, y_list = [], []
print('Building augmented HOG dataset...')
for idx, (img, label) in enumerate(samples):
    X_list.append(to_feature_vec(img))
    y_list.append(label)
    for _ in range(AUG_PER_IMAGE):
        X_list.append(to_feature_vec(aug(img)))
        y_list.append(label)
    if (idx + 1) % 20 == 0:
        print(f'  {idx+1}/{len(samples)}')

X = np.stack(X_list).astype(np.float32)
y = np.asarray(y_list, dtype=np.int64)
print(f'\nX: {X.shape}  y: {y.shape}')

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=VAL_SPLIT, stratify=y, random_state=SEED
)
print(f'Train: {len(X_train)}   Val: {len(X_val)}')

## 5. Models & Hyperparameter Search Spaces

In [ ]:
SEARCH_CONFIGS = {
    'Random Forest': {
        'estimator': RandomForestClassifier(
            class_weight='balanced', random_state=SEED, n_jobs=-1
        ),
        'param_dist': {
            'n_estimators':      randint(300, 900),
            'max_depth':         [None, 20, 30, 40],
            'max_features':      ['sqrt', 'log2', 0.2, 0.3],
            'min_samples_leaf':  randint(1, 4),
            'min_samples_split': randint(2, 7),
        },
    },
    'SVM (RBF)': {
        'estimator': Pipeline([
            ('scaler', StandardScaler()),
            ('svc', SVC(kernel='rbf', probability=True, random_state=SEED)),
        ]),
        'param_dist': {
            'svc__C':     loguniform(1, 300),
            'svc__gamma': ['scale', 'auto'] + list(np.logspace(-3, 0, 8)),
        },
    },
    'MLP': {
        'estimator': Pipeline([
            ('scaler', StandardScaler()),
            ('mlp', MLPClassifier(
                activation='relu', max_iter=500,
                early_stopping=True, validation_fraction=0.1,
                n_iter_no_change=20, random_state=SEED,
            )),
        ]),
        'param_dist': {
            'mlp__hidden_layer_sizes': [
                (512,), (256, 128), (512, 256),
                (256, 128, 64), (512, 256, 128),
            ],
            'mlp__alpha':              loguniform(1e-5, 1e-2),
            'mlp__learning_rate_init': loguniform(1e-4, 5e-3),
        },
    },
    'KNN': {
        'estimator': Pipeline([
            ('scaler', StandardScaler()),
            ('knn', KNeighborsClassifier(n_jobs=-1)),
        ]),
        'param_dist': {
            'knn__n_neighbors': randint(3, 16),
            'knn__metric':      ['euclidean', 'cosine', 'manhattan'],
            'knn__weights':     ['uniform', 'distance'],
        },
    },
}

BASELINES = {
    'Decision Tree': DecisionTreeClassifier(random_state=SEED),
    'Naive Bayes':   GaussianNB(),
}
print('Models defined.')

## 6. Hyperparameter Search (RandomizedSearchCV)

In [ ]:
skf_inner   = StratifiedKFold(n_splits=3, shuffle=True, random_state=SEED)
best_models = {}

for name, cfg in SEARCH_CONFIGS.items():
    print(f'\n== {name} — {N_SEARCH_ITER} iter ==')
    t0 = time.time()
    search = RandomizedSearchCV(
        cfg['estimator'], cfg['param_dist'],
        n_iter=N_SEARCH_ITER, cv=skf_inner,
        scoring='f1_macro', n_jobs=-1,
        random_state=SEED, refit=True, verbose=1,
    )
    search.fit(X_train, y_train)
    print(f'  Best CV F1: {search.best_score_:.4f}  ({time.time()-t0:.0f}s)')
    print(f'  Params: {search.best_params_}')
    best_models[name] = search.best_estimator_

print('\n-- Baselines --')
for name, clf in BASELINES.items():
    clf.fit(X_train, y_train)
    best_models[name] = clf
    print(f'  {name}: done')

print(f'\nAll {len(best_models)} models ready.')

## 7. Evaluation — Accuracy · Precision · Recall · F1-score

In [ ]:
results = {}
for name, clf in best_models.items():
    y_pred = clf.predict(X_val)
    results[name] = {
        'Accuracy':  accuracy_score(y_val, y_pred),
        'Precision': precision_score(y_val, y_pred, average='macro', zero_division=0),
        'Recall':    recall_score(y_val, y_pred,    average='macro', zero_division=0),
        'F1-score':  f1_score(y_val, y_pred,        average='macro', zero_division=0),
    }

df_results = pd.DataFrame(results).T.sort_values('F1-score', ascending=False)
print('\n=== 7. Evaluation Results ===')
print(df_results.applymap(lambda x: f'{x:.4f}').to_string())

In [ ]:
# Bar chart
metrics = ['Accuracy', 'Precision', 'Recall', 'F1-score']
colors  = ['#2563eb', '#16a34a', '#d97706', '#dc2626']
names   = list(df_results.index)
x, width = np.arange(len(names)), 0.2

fig, ax = plt.subplots(figsize=(14, 5))
for i, (m, c) in enumerate(zip(metrics, colors)):
    vals = [df_results.loc[n, m] for n in names]
    bars = ax.bar(x + i*width, vals, width, label=m, color=c, alpha=0.85)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.005,
                f'{v:.2f}', ha='center', va='bottom', fontsize=7)
ax.set_xticks(x + width*1.5)
ax.set_xticklabels(names, rotation=15, ha='right')
ax.set_ylim(0, 1.12); ax.set_ylabel('Score')
ax.set_title('Optimized Model Comparison — Thai Digits 51–55 (HOG features)')
ax.legend(loc='lower right'); ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('comparison_optimized.png', dpi=150)
plt.show()

In [ ]:
# Confusion matrices
n = len(best_models)
fig, axes = plt.subplots((n+2)//3, 3, figsize=(15, ((n+2)//3)*5))
axes = axes.flatten()
for ax, (name, clf) in zip(axes, best_models.items()):
    y_pred = clf.predict(X_val)
    cm = confusion_matrix(y_val, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=ax, cbar=False)
    ax.set_title(f'{name}\nAcc={accuracy_score(y_val, y_pred):.3f}', fontsize=10)
    ax.set_xlabel('Predicted'); ax.set_ylabel('True')
for ax in axes[n:]: ax.set_visible(False)
plt.suptitle('Confusion Matrices — Optimized Pipeline', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('confusion_matrices_optimized.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# 5-Fold CV
print('=== 5-Fold CV ===')
skf_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
for name, clf in best_models.items():
    scores = cross_val_score(clf, X, y, cv=skf_cv, scoring='accuracy', n_jobs=-1)
    print(f'{name:<20}  {scores.mean():.4f} +/- {scores.std():.4f}')

## 8. Save Best Model

In [ ]:
best_name = df_results.index[0]
best_clf  = best_models[best_name]
print(f'Best: {best_name}  F1={results[best_name]["F1-score"]:.4f}')

slug     = best_name.lower().replace(' ','_').replace('(','').replace(')','').replace('/','_')
filename = f'best_model_{slug}.joblib'

joblib.dump({
    'model':        best_clf,
    'classes':      CLASS_NAMES,
    'image_shape':  (IMG_SIZE, IMG_SIZE),
    'input_type':   'hog',
    'hog_params':   {'orientations': 9, 'pixels_per_cell': (4,4),
                     'cells_per_block': (2,2), 'block_norm': 'L2-Hys'},
    'preprocessing': 'gaussian_denoise(sigma=0.8)->otsu_center->resize_28->normalize[0,1]->HOG_1296d',
    'model_name':   best_name,
}, filename, compress=3)
print(f'Saved {filename}  ({os.path.getsize(filename)/1024:.1f} KB)')

from google.colab import files as gfiles
gfiles.download(filename)

## 9. (Optional) Save All Models as ZIP

In [ ]:
import zipfile as zf
os.makedirs('all_models', exist_ok=True)
saved = []
for name, clf in best_models.items():
    slug  = name.lower().replace(' ','_').replace('(','').replace(')','').replace('/','_')
    fname = f'all_models/{slug}.joblib'
    joblib.dump({
        'model': clf, 'classes': CLASS_NAMES,
        'image_shape': (IMG_SIZE, IMG_SIZE),
        'input_type':  'hog',
        'hog_params':  {'orientations': 9, 'pixels_per_cell': (4,4),
                        'cells_per_block': (2,2), 'block_norm': 'L2-Hys'},
        'preprocessing': 'gaussian_denoise(sigma=0.8)->otsu_center->resize_28->normalize[0,1]->HOG_1296d',
        'model_name': name,
    }, fname, compress=3)
    saved.append(fname)
    print(f'  {fname}  ({os.path.getsize(fname)/1024:.1f} KB)')

with zf.ZipFile('all_models_optimized.zip', 'w') as z:
    for f in saved: z.write(f)
gfiles.download('all_models_optimized.zip')
print('Done.')